# Fase 4.2 (CRISP-DM): Modelado Supervisado — Predicción de ETA y Pruebas por Nivel

**Metodología:** CRISP-DM — Fase 4.2 (*Modeling — Clustering + Regresión*)  
**Entradas:** `data/03_processed/train_final.parquet` (`36,467 × 49`) y `data/03_processed/test_final.parquet` (`9,117 × 49`)  
**Salidas:** `models/preprocessor_reg.joblib`, `models/best_eta_model.joblib`, 

## 1. Imports, Definiciones del Pipeline y Verificación de Particiones Finales

Se configuran los parámetros y la arquitectura híbrida (`ClusterSpecializedHybridRegressor`) directamente en el notebok y se verifica la ausencia total de valores nulos y fuga de datos entre `train` y `test`.


In [ ]:
from __future__ import annotations

import time
from pathlib import Path
from typing import Any

import scipy.stats as stats  # prevenir bloqueos DLL en Windows antes de sklearn
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_style('whitegrid')

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RANDOM_STATE = 42
TARGET_COL = "Time_taken (min)"
_DEFAULT_MODELS_DIR = ROOT_DIR / "models"

CLUSTER_NAMES: dict[int, str] = {
    0: "Express urbano matutino",
    1: "Operación estándar nocturna",
    2: "Cuello de botella operativo",
}

COLS_NIVEL_1: list[str] = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Vehicle_condition",
    "multiple_deliveries",
    "Road_traffic_density",
    "Festival_Yes",
    "City_ord",
    "City_Semi-Urban",
    "City_Urban",
    "City_Unknown",
    "Weather_conditions",
    "Weather_Cloudy",
    "Weather_Fog",
    "Weather_Sandstorms",
    "Weather_Stormy",
    "Weather_Windy",
    "Type_of_order_Drinks",
    "Type_of_order_Meal",
    "Type_of_order_Snack",
    "Type_of_vehicle_motorcycle",
    "Type_of_vehicle_scooter",
    "Type_of_vehicle_electric_scooter",
]

COLS_NIVEL_2: list[str] = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "dist_haversine_km",
    "dist_manhattan_km",
    "Tiempo_preparacion_min",
    "dia_semana",
    "es_fin_semana",
    "hora_pedido",
    "hora_sin",
    "hora_cos",
    "Franja_Manana",
    "Franja_Almuerzo_Pico",
    "Franja_Tarde_Valle",
    "Franja_Cena_Pico",
    "Vehicle_condition",
    "multiple_deliveries",
    "Road_traffic_density",
    "Festival_Yes",
    "City_ord",
    "City_Semi-Urban",
    "City_Urban",
    "City_Unknown",
    "Weather_conditions",
    "Weather_Cloudy",
    "Weather_Fog",
    "Weather_Sandstorms",
    "Weather_Stormy",
    "Weather_Windy",
    "Type_of_order_Drinks",
    "Type_of_order_Meal",
    "Type_of_order_Snack",
    "Type_of_vehicle_motorcycle",
    "Type_of_vehicle_scooter",
    "Type_of_vehicle_electric_scooter",
]

CLUSTER_FEATURE_COLS: list[str] = [
    "cluster_label",
    "dist_centroide_1",
    "dist_centroide_2",
    "dist_centroide_3",
    "prob_pertenencia_1",
    "prob_pertenencia_2",
    "prob_pertenencia_3",
]

COLS_NIVEL_3: list[str] = COLS_NIVEL_2 + CLUSTER_FEATURE_COLS

CONTINUOUS_SCALE_COLS: list[str] = [
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "dist_haversine_km",
    "dist_manhattan_km",
    "Tiempo_preparacion_min",
    "dia_semana",
    "hora_pedido",
    "hora_sin",
    "hora_cos",
    "dist_centroide_1",
    "dist_centroide_2",
    "dist_centroide_3",
    "prob_pertenencia_1",
    "prob_pertenencia_2",
    "prob_pertenencia_3",
]


def get_feature_cols(df: pd.DataFrame, level: int = 3) -> list[str]:
    """Devuelve las columnas presentes en `df` correspondientes al nivel de la prueba."""
    mapping = {1: COLS_NIVEL_1, 2: COLS_NIVEL_2, 3: COLS_NIVEL_3}
    if level not in mapping:
        raise ValueError(f"Nivel inválido: {level}. Use 1, 2 o 3.")
    return [col for col in mapping[level] if col in df.columns]


def build_regression_preprocessor(feature_cols: list[str] | None = None) -> ColumnTransformer:
    """Construye un `ColumnTransformer` reproducible para las variables de regresión."""
    cols = list(feature_cols) if feature_cols is not None else list(COLS_NIVEL_3)
    scale_cols = [c for c in CONTINUOUS_SCALE_COLS if c in cols]
    pass_cols = [c for c in cols if c not in scale_cols]

    transformers = []
    if scale_cols:
        transformers.append(("num_scaler", StandardScaler(), scale_cols))
    if pass_cols:
        transformers.append(("pass_features", "passthrough", pass_cols))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=False,
    )


def save_preprocessor(
    preprocessor: ColumnTransformer, path: str | Path | None = None
) -> Path:
    """Serializa el `ColumnTransformer` en `models/preprocessor_reg.joblib`."""
    target_path = (
        _DEFAULT_MODELS_DIR / "preprocessor_reg.joblib" if path is None else Path(path)
    )
    target_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(preprocessor, target_path)
    return target_path


class ClusterSpecializedHybridRegressor(BaseEstimator, RegressorMixin):
    """Regresor híbrido (Estrategia Cluster + Regresion): combina modelo global y especialistas por cluster.

    El componente global opera sobre las variables limpias y de ingeniería (`global_cols`)
    para evitar degradar particiones por redundancia radial, mientras que los
    sub-modelos especializados aprenden la superficie no lineal específica dentro de cada
    régimen operativo (`cluster_label`) utilizando además las distancias continuas a los
    centroides y probabilidades de pertenencia (`cluster_cols`).
    """

    def __init__(
        self,
        global_params: dict[str, Any] | None = None,
        cluster_params: dict[str, Any] | None = None,
        global_weight: float = 0.70,
        cluster_col: str = "cluster_label",
        global_cols: list[str] | None = None,
        cluster_cols: list[str] | None = None,
        random_state: int = RANDOM_STATE,
    ) -> None:
        self.global_params = global_params
        self.cluster_params = cluster_params
        self.global_weight = float(global_weight)
        self.cluster_col = cluster_col
        self.global_cols = global_cols
        self.cluster_cols = cluster_cols
        self.random_state = int(random_state)

    def _resolve_params(self) -> tuple[dict[str, Any], dict[str, Any]]:
        default_global = {
            "max_iter": 300,
            "max_depth": 10,
            "max_leaf_nodes": 25,
            "min_samples_leaf": 60,
            "learning_rate": 0.05,
            "l2_regularization": 2.0,
            "random_state": self.random_state,
        }
        default_cluster = {
            "max_iter": 250,
            "max_depth": 8,
            "max_leaf_nodes": 20,
            "min_samples_leaf": 40,
            "learning_rate": 0.04,
            "l2_regularization": 5.0,
            "random_state": self.random_state,
        }
        g_params = {**default_global, **(self.global_params or {})}
        c_params = {**default_cluster, **(self.cluster_params or {})}
        return g_params, c_params

    def _prepare_indices(self, X: pd.DataFrame | np.ndarray) -> None:
        if isinstance(X, pd.DataFrame):
            cols = list(X.columns)
            self.feature_names_in_ = np.asarray(cols, dtype=object)
            self.n_features_in_ = len(cols)
        else:
            cols = list(self.cluster_cols or COLS_NIVEL_3)
            self.feature_names_in_ = np.asarray(cols, dtype=object)
            self.n_features_in_ = X.shape[1]

        g_target = list(self.global_cols or COLS_NIVEL_2)
        c_target = [
            c
            for c in (self.cluster_cols or COLS_NIVEL_3)
            if c != self.cluster_col
        ]

        self.global_indices_ = [cols.index(c) for c in g_target if c in cols]
        self.cluster_indices_ = [cols.index(c) for c in c_target if c in cols]
        if not self.global_indices_:
            self.global_indices_ = list(range(self.n_features_in_))
        if not self.cluster_indices_:
            self.cluster_indices_ = list(range(self.n_features_in_))
        if self.cluster_col not in cols:
            raise ValueError(
                f"La columna '{self.cluster_col}' es requerida por ClusterSpecializedHybridRegressor."
            )
        self.cluster_col_idx_ = cols.index(self.cluster_col)

    def fit(self, X: pd.DataFrame | np.ndarray, y: pd.Series | np.ndarray) -> "ClusterSpecializedHybridRegressor":
        self._prepare_indices(X)
        arr = np.asarray(X, dtype=float)
        y_arr = np.asarray(y, dtype=float)

        g_params, c_params = self._resolve_params()
        self.global_model_ = HistGradientBoostingRegressor(**g_params)
        self.global_model_.fit(arr[:, self.global_indices_], y_arr)

        clusters = np.round(arr[:, self.cluster_col_idx_]).astype(int)
        self.clusters_ = np.sort(np.unique(clusters))
        self.cluster_models_: dict[int, HistGradientBoostingRegressor] = {}

        for cluster_id in self.clusters_:
            mask = clusters == cluster_id
            model_k = HistGradientBoostingRegressor(**c_params)
            model_k.fit(arr[mask][:, self.cluster_indices_], y_arr[mask])
            self.cluster_models_[int(cluster_id)] = model_k

        return self

    def predict(self, X: pd.DataFrame | np.ndarray) -> np.ndarray:
        arr = np.asarray(X, dtype=float)
        pred_global = self.global_model_.predict(arr[:, self.global_indices_])

        clusters = np.round(arr[:, self.cluster_col_idx_]).astype(int)
        pred_cluster = pred_global.copy()

        for cluster_id, model_k in self.cluster_models_.items():
            mask = clusters == cluster_id
            if np.any(mask):
                pred_cluster[mask] = model_k.predict(arr[mask][:, self.cluster_indices_])

        w = self.global_weight
        return w * pred_global + (1.0 - w) * pred_cluster


def compute_metrics(
    y_true: np.ndarray | pd.Series, y_pred: np.ndarray | pd.Series
) -> dict[str, float]:
    """Calcula MAE, RMSE, MAPE (%), MedAE, R2 y porcentaje de errores severos (>10 min)."""
    yt = np.asarray(y_true, dtype=float)
    yp = np.asarray(y_pred, dtype=float)
    abs_err = np.abs(yt - yp)
    return {
        "MAE": float(mean_absolute_error(yt, yp)),
        "RMSE": float(root_mean_squared_error(yt, yp)),
        "MAPE_pct": float(mean_absolute_percentage_error(yt, yp) * 100.0),
        "MedAE": float(median_absolute_error(yt, yp)),
        "R2": float(r2_score(yt, yp)),
        "Error_Gt_10min_pct": float(np.mean(abs_err > 10.0) * 100.0),
        "Error_Gt_8min_pct": float(np.mean(abs_err > 8.0) * 100.0),
    }


def _make_candidate_estimators(random_state: int = RANDOM_STATE) -> dict[str, Any]:
    """Construye las 4 familias comparativas requeridas."""
    return {
        "Ridge": lambda: Pipeline(
            [
                ("scaler", StandardScaler()),
                ("regressor", Ridge(alpha=10.0)),
            ]
        ),
        "RandomForest": lambda: RandomForestRegressor(
            n_estimators=60,
            max_depth=14,
            min_samples_leaf=10,
            max_samples=0.65,
            max_features=0.7,
            n_jobs=1,
            random_state=random_state,
        ),
        "XGBoost_GBDT": lambda: GradientBoostingRegressor(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.10,
            subsample=0.75,
            max_features=0.8,
            min_samples_leaf=30,
            random_state=random_state,
        ),
        "LightGBM_HistGBM": lambda: HistGradientBoostingRegressor(
            max_iter=300,
            max_depth=10,
            max_leaf_nodes=25,
            min_samples_leaf=60,
            learning_rate=0.05,
            l2_regularization=2.0,
            random_state=random_state,
        ),
    }


def _cross_validate_model(
    model_factory: Any,
    X: pd.DataFrame,
    y: pd.Series,
    cv: KFold,
) -> dict[str, float]:
    """Ejecuta validación cruzada K-Fold en `train` retornando promedio y desviación estándar."""
    fold_metrics = []
    for tr_idx, va_idx in cv.split(X):
        model = model_factory()
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        preds = model.predict(X.iloc[va_idx])
        fold_metrics.append(compute_metrics(y.iloc[va_idx], preds))

    df_folds = pd.DataFrame(fold_metrics)
    return {
        "CV_MAE_mean": float(df_folds["MAE"].mean()),
        "CV_MAE_std": float(df_folds["MAE"].std()),
        "CV_RMSE_mean": float(df_folds["RMSE"].mean()),
        "CV_RMSE_std": float(df_folds["RMSE"].std()),
        "CV_R2_mean": float(df_folds["R2"].mean()),
    }


def run_ablation_study(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    cv_splits: int = 5,
    random_state: int = RANDOM_STATE,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Ejecuta el estudio de pruebas completo en 3 niveles con CV 5-Fold y evaluación en test."""
    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    y_train = df_train[TARGET_COL]
    y_test = df_test[TARGET_COL]
    factories = _make_candidate_estimators(random_state=random_state)

    levels = [
        ("Nivel 1 (Baseline)", 1),
        ("Nivel 2 (Feature Eng.)", 2),
        ("Nivel 3 (A: FE + Clusters)", 3),
    ]

    rows: list[dict[str, Any]] = []
    key_predictions: dict[str, np.ndarray] = {}

    for level_name, level_num in levels:
        cols = get_feature_cols(df_train, level=level_num)
        X_train = df_train[cols]
        X_test = df_test[cols]

        for algo_name, factory in factories.items():
            t0 = time.perf_counter()
            cv_stats = _cross_validate_model(factory, X_train, y_train, cv)
            model = factory()
            model.fit(X_train, y_train)
            preds_test = model.predict(X_test)
            elapsed = time.perf_counter() - t0

            test_stats = compute_metrics(y_test, preds_test)
            row = {
                "Nivel": level_name,
                "Nivel_Num": level_num,
                "Estrategia": "Estándar",
                "Modelo": algo_name,
                "N_Features": len(cols),
                "CV_MAE_mean": round(cv_stats["CV_MAE_mean"], 4),
                "CV_MAE_std": round(cv_stats["CV_MAE_std"], 4),
                "CV_RMSE_mean": round(cv_stats["CV_RMSE_mean"], 4),
                "CV_R2_mean": round(cv_stats["CV_R2_mean"], 4),
                "Test_MAE": round(test_stats["MAE"], 4),
                "Test_RMSE": round(test_stats["RMSE"], 4),
                "Test_MAPE_pct": round(test_stats["MAPE_pct"], 2),
                "Test_R2": round(test_stats["R2"], 4),
                "Test_Err_Gt_10min_pct": round(test_stats["Error_Gt_10min_pct"], 2),
                "Tiempo_s": round(elapsed, 2),
            }
            rows.append(row)

            if algo_name == "LightGBM_HistGBM":
                key_predictions[f"lvl{level_num}_{algo_name}"] = preds_test

            if verbose:
                print(
                    f"[{level_name} | {algo_name}] "
                    f"CV MAE={row['CV_MAE_mean']:.4f}±{row['CV_MAE_std']:.4f} | "
                    f"Test MAE={row['Test_MAE']:.4f} RMSE={row['Test_RMSE']:.4f} R2={row['Test_R2']:.4f} "
                    f"({row['Tiempo_s']}s)",
                    flush=True,
                )

    # Nivel 3 Híbrido (A + C): Modelo Global + Especialistas por Cluster
    cols_lvl3 = get_feature_cols(df_train, level=3)
    X_train_3 = df_train[cols_lvl3]
    X_test_3 = df_test[cols_lvl3]
    hybrid_factory = lambda: ClusterSpecializedHybridRegressor(
        global_weight=0.70, random_state=random_state
    )
    t0 = time.perf_counter()
    cv_stats_h = _cross_validate_model(hybrid_factory, X_train_3, y_train, cv)
    hybrid_base = hybrid_factory().fit(X_train_3, y_train)
    preds_hybrid_base = hybrid_base.predict(X_test_3)
    elapsed_h = time.perf_counter() - t0
    test_stats_h = compute_metrics(y_test, preds_hybrid_base)

    row_h = {
        "Nivel": "Nivel 3 (Híbrido A+C)",
        "Nivel_Num": 3,
        "Estrategia": "Global + Especialista Cluster",
        "Modelo": "Hybrid_Cluster_HistGBM",
        "N_Features": len(cols_lvl3),
        "CV_MAE_mean": round(cv_stats_h["CV_MAE_mean"], 4),
        "CV_MAE_std": round(cv_stats_h["CV_MAE_std"], 4),
        "CV_RMSE_mean": round(cv_stats_h["CV_RMSE_mean"], 4),
        "CV_R2_mean": round(cv_stats_h["CV_R2_mean"], 4),
        "Test_MAE": round(test_stats_h["MAE"], 4),
        "Test_RMSE": round(test_stats_h["RMSE"], 4),
        "Test_MAPE_pct": round(test_stats_h["MAPE_pct"], 2),
        "Test_R2": round(test_stats_h["R2"], 4),
        "Test_Err_Gt_10min_pct": round(test_stats_h["Error_Gt_10min_pct"], 2),
        "Tiempo_s": round(elapsed_h, 2),
    }
    rows.append(row_h)
    key_predictions["lvl3_Hybrid_Base"] = preds_hybrid_base

    if verbose:
        print(
            f"[Nivel 3 (Híbrido A+C) | Hybrid_Cluster_HistGBM] "
            f"CV MAE={row_h['CV_MAE_mean']:.4f}±{row_h['CV_MAE_std']:.4f} | "
            f"Test MAE={row_h['Test_MAE']:.4f} RMSE={row_h['Test_RMSE']:.4f} R2={row_h['Test_R2']:.4f} "
            f"({row_h['Tiempo_s']}s)",
            flush=True,
        )

    return pd.DataFrame(rows), key_predictions


def tune_and_fit_best_model(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    cv_splits: int = 5,
    random_state: int = RANDOM_STATE,
    model_path: str | Path | None = None,
    preprocessor_path: str | Path | None = None,
    verbose: bool = True,
) -> tuple[ClusterSpecializedHybridRegressor, pd.DataFrame, dict[str, Any]]:
    """Optimiza hiperparámetros por CV en `train` y exporta el mejor modelo y preprocesador."""
    cols_lvl3 = get_feature_cols(df_train, level=3)
    X_train = df_train[cols_lvl3]
    y_train = df_train[TARGET_COL]
    X_test = df_test[cols_lvl3]
    y_test = df_test[TARGET_COL]

    # Construir, ajustar y serializar el ColumnTransformer (sobre train exclusivamente)
    preprocessor = build_regression_preprocessor(cols_lvl3)
    preprocessor.fit(X_train)
    saved_prep_path = save_preprocessor(preprocessor, path=preprocessor_path)

    cv = KFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    candidates = [
        {
            "Config": "Base_Leaf25_w0.70",
            "global_params": {
                "max_iter": 300,
                "max_depth": 10,
                "max_leaf_nodes": 25,
                "min_samples_leaf": 60,
                "learning_rate": 0.05,
                "l2_regularization": 2.0,
            },
            "cluster_params": {
                "max_iter": 250,
                "max_depth": 8,
                "max_leaf_nodes": 20,
                "min_samples_leaf": 40,
                "learning_rate": 0.04,
                "l2_regularization": 5.0,
            },
            "global_weight": 0.70,
        },
        {
            "Config": "Reg_Leaf25_w0.75",
            "global_params": {
                "max_iter": 350,
                "max_depth": 10,
                "max_leaf_nodes": 28,
                "min_samples_leaf": 55,
                "learning_rate": 0.045,
                "l2_regularization": 2.5,
            },
            "cluster_params": {
                "max_iter": 280,
                "max_depth": 8,
                "max_leaf_nodes": 22,
                "min_samples_leaf": 40,
                "learning_rate": 0.04,
                "l2_regularization": 4.0,
            },
            "global_weight": 0.75,
        },
        {
            "Config": "Tuned_Leaf31_w0.80",
            "global_params": {
                "max_iter": 350,
                "max_depth": 10,
                "max_leaf_nodes": 31,
                "min_samples_leaf": 50,
                "learning_rate": 0.04,
                "l2_regularization": 3.0,
            },
            "cluster_params": {
                "max_iter": 300,
                "max_depth": 9,
                "max_leaf_nodes": 25,
                "min_samples_leaf": 35,
                "learning_rate": 0.04,
                "l2_regularization": 3.0,
            },
            "global_weight": 0.80,
        },
    ]

    tuning_rows = []
    for cfg in candidates:
        factory = lambda c=cfg: ClusterSpecializedHybridRegressor(
            global_params=c["global_params"],
            cluster_params=c["cluster_params"],
            global_weight=c["global_weight"],
            random_state=random_state,
        )
        cv_res = _cross_validate_model(factory, X_train, y_train, cv)
        tuning_rows.append(
            {
                "Config": cfg["Config"],
                "global_weight": cfg["global_weight"],
                "global_max_leaf_nodes": cfg["global_params"]["max_leaf_nodes"],
                "global_min_samples_leaf": cfg["global_params"]["min_samples_leaf"],
                "cluster_max_leaf_nodes": cfg["cluster_params"]["max_leaf_nodes"],
                "CV_MAE_mean": round(cv_res["CV_MAE_mean"], 4),
                "CV_MAE_std": round(cv_res["CV_MAE_std"], 4),
                "CV_RMSE_mean": round(cv_res["CV_RMSE_mean"], 4),
                "CV_R2_mean": round(cv_res["CV_R2_mean"], 4),
                "_cfg": cfg,
            }
        )
        if verbose:
            print(
                f"[Tuning {cfg['Config']}] CV MAE={cv_res['CV_MAE_mean']:.4f} | "
                f"CV RMSE={cv_res['CV_RMSE_mean']:.4f} | CV R2={cv_res['CV_R2_mean']:.4f}",
                flush=True,
            )

    df_tuning = pd.DataFrame(tuning_rows).sort_values("CV_MAE_mean").reset_index(drop=True)
    best_cfg = df_tuning.loc[0, "_cfg"]
    df_tuning_clean = df_tuning.drop(columns=["_cfg"])

    best_model = ClusterSpecializedHybridRegressor(
        global_params=best_cfg["global_params"],
        cluster_params=best_cfg["cluster_params"],
        global_weight=best_cfg["global_weight"],
        random_state=random_state,
    )
    best_model.fit(X_train, y_train)

    # También se entrena el equivalente Nivel 2 con exactamente los hiperparámetros globales ganadores
    # para una comparación pareada estrictamente justa.
    cols_lvl2 = get_feature_cols(df_train, level=2)
    tuned_lvl2_model = HistGradientBoostingRegressor(
        **{**best_cfg["global_params"], "random_state": random_state}
    )
    tuned_lvl2_model.fit(df_train[cols_lvl2], y_train)

    preds_lvl3_tuned = best_model.predict(X_test)
    preds_lvl2_tuned = tuned_lvl2_model.predict(df_test[cols_lvl2])

    target_model_path = (
        _DEFAULT_MODELS_DIR / "best_eta_model.joblib"
        if model_path is None
        else Path(model_path)
    )
    target_model_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_model, target_model_path)

    artifacts_info = {
        "best_config_name": best_cfg["Config"],
        "best_config": best_cfg,
        "model_path": str(target_model_path),
        "preprocessor_path": str(saved_prep_path),
        "test_metrics_lvl3_tuned": compute_metrics(y_test, preds_lvl3_tuned),
        "test_metrics_lvl2_tuned": compute_metrics(y_test, preds_lvl2_tuned),
        "preds_lvl3_tuned": preds_lvl3_tuned,
        "preds_lvl2_tuned": preds_lvl2_tuned,
    }
    return best_model, df_tuning_clean, artifacts_info


df_train = pd.read_parquet('../data/03_processed/train_final.parquet')
df_test = pd.read_parquet('../data/03_processed/test_final.parquet')

print(f'Train final: {df_train.shape} | Nulos: {int(df_train.isna().sum().sum())}')
print(f'Test  final: {df_test.shape}  | Nulos: {int(df_test.isna().sum().sum())}')
print(
    f'Variable objetivo: {TARGET_COL} | '
    f'Rango train: [{df_train[TARGET_COL].min()}, {df_train[TARGET_COL].max()}] min'
)


Train final: (36467, 49) | Nulos: 0
Test  final: (9117, 49)  | Nulos: 0
Variable objetivo: Time_taken (min) | Rango train: [10, 54] min


## 2. Definición de Conjuntos de Prueba y Preprocesador (`ColumnTransformer`)

El experimento controla tres niveles progresivo:
1. **Nivel 1 (Baseline):** Variables originales limpias (incluyendo coordenadas crudas y codificación nominal/ordinal básica), **sin** distancias geométricas, **sin** tiempo de preparación, **sin** atributos temporales/cíclicos y **sin** variables de *clustering*.
2. **Nivel 2 (Feature Engineering):** Sustituye latitud/longitud crudas por `dist_haversine_km`, `dist_manhattan_km`, `Tiempo_preparacion_min`, atributos de calendario/cíclicos (`dia_semana`, `es_fin_semana`, `hora_pedido`, `hora_sin`, `hora_cos`) y franjas horarias (`Franja_*`).
3. **Nivel 3 (Híbrido):** Incorpora `cluster_label`, distancias euclidianas a los centroides (`dist_centroide_1..3`) y probabilidades de pertenencia (`prob_pertenencia_1..3`), evaluando tanto su inserción tabular directa  como la arquitectura especializada (**Híbrido Cluster + Regresion**: `ClusterSpecializedHybridRegressor`).

Además, se ajusta un `ColumnTransformer` exclusivamente sobre `train` y se exporta en `models/preprocessor_reg.joblib`.

In [ ]:
cols_1 = get_feature_cols(df_train, level=1)
cols_2 = get_feature_cols(df_train, level=2)
cols_3 = get_feature_cols(df_train, level=3)

print(f'Nivel 1 (Baseline):           {len(cols_1)} variables')
print(f'Nivel 2 (Feature Eng.):       {len(cols_2)} variables')
print(f'Nivel 3 (Híbrido A + C):      {len(cols_3)} variables (+7 de clustering)')

preprocessor_reg = build_regression_preprocessor(cols_3)
X_train_scaled = preprocessor_reg.fit_transform(df_train[cols_3])
X_test_scaled = preprocessor_reg.transform(df_test[cols_3])
prep_path = save_preprocessor(preprocessor_reg, path='../models/preprocessor_reg.joblib')

print(f'Preprocesador serializado en: {prep_path}')
print(f'Dimensiones transformadas -> Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}')

Nivel 1 (Baseline):           26 variables
Nivel 2 (Feature Eng.):       34 variables
Nivel 3 (Híbrido A + C):      41 variables (+7 de clustering)
Preprocesador serializado en: ..\models\preprocessor_reg.joblib
Dimensiones transformadas -> Train: (36467, 41) | Test: (9117, 41)


## 3. Estudio de Pruebas (Validación Cruzada 5-Fold en `train` + `test`)

Se comparan cuatro familias de regresión a lo largo de los tres niveles:
- **Ridge** (regresión lineal con penalización L2 y `StandardScaler`)
- **RandomForest** (ensamble *Bagging* no lineal)
- **XGBoost_GBDT** (ensamble *Gradient Boosting* secuencial)
- **LightGBM_HistGBM** (ensamble *Histogram-based Gradient Boosting*)
- **Hybrid_Cluster_HistGBM** (`ClusterSpecializedHybridRegressor`: componente global + sub-modelos especializados por cluster operativo)


In [3]:
df_ablation, key_preds = run_ablation_study(
    df_train, df_test, cv_splits=5, random_state=RANDOM_STATE, verbose=True
)

reports_dir = Path('../reports')
reports_dir.mkdir(parents=True, exist_ok=True)
df_ablation.to_csv(reports_dir / 'ablacion_resultados.csv', index=False)

display(df_ablation.sort_values(['Nivel_Num', 'Test_MAE']).reset_index(drop=True))

[Nivel 1 (Baseline) | Ridge] CV MAE=5.0093±0.0234 | Test MAE=4.9460 RMSE=6.2174 R2=0.5609 (0.21s)


[Nivel 1 (Baseline) | RandomForest] CV MAE=3.9261±0.0298 | Test MAE=3.8352 RMSE=4.9050 R2=0.7267 (16.43s)


[Nivel 1 (Baseline) | XGBoost_GBDT] CV MAE=3.8437±0.0380 | Test MAE=3.7593 RMSE=4.7649 R2=0.7421 (24.11s)


[Nivel 1 (Baseline) | LightGBM_HistGBM] CV MAE=3.6974±0.0300 | Test MAE=3.6086 RMSE=4.5562 R2=0.7642 (6.1s)


[Nivel 2 (Feature Eng.) | Ridge] CV MAE=4.8116±0.0365 | Test MAE=4.7869 RMSE=6.0377 R2=0.5859 (0.28s)


[Nivel 2 (Feature Eng.) | RandomForest] CV MAE=3.1465±0.0333 | Test MAE=3.0954 RMSE=3.8769 R2=0.8293 (17.49s)


... (4 outputs left out)

[Nivel 3 (A: FE + Clusters) | XGBoost_GBDT] CV MAE=3.2967±0.0273 | Test MAE=3.2502 RMSE=4.0758 R2=0.8113 (58.52s)


[Nivel 3 (A: FE + Clusters) | LightGBM_HistGBM] CV MAE=3.2032±0.0169 | Test MAE=3.1437 RMSE=3.9455 R2=0.8232 (5.53s)


[Nivel 3 (Híbrido A+C) | Hybrid_Cluster_HistGBM] CV MAE=3.1655±0.0200 | Test MAE=3.1002 RMSE=3.9002 R2=0.8272 (13.86s)


,Nivel,Nivel_Num,Estrategia,Modelo,N_Features,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_R2_mean,Test_MAE,Test_RMSE,Test_MAPE_pct,Test_R2,Test_Err_Gt_10min_pct,Tiempo_s
0,Nivel 1 (Baseline),1,Estándar,LightGBM_HistGBM,26,3.6974,0.0300,4.6830,0.7510,3.6086,4.5562,15.84,0.7642,3.10,6.10
1,Nivel 1 (Baseline),1,Estándar,XGBoost_GBDT,26,3.8437,0.0380,4.8742,0.7302,3.7593,4.7649,16.58,0.7421,3.75,24.11
2,Nivel 1 (Baseline),1,Estándar,RandomForest,26,3.9261,0.0298,5.0280,0.7129,3.8352,4.9050,16.70,0.7267,4.82,16.43
3,Nivel 1 (Baseline),1,Estándar,Ridge,26,5.0093,0.0234,6.2952,0.5500,4.9460,6.2174,21.90,0.5609,10.33,0.21
4,Nivel 2 (Feature Eng.),2,Estándar,RandomForest,34,3.1465,0.0333,3.9604,0.8219,3.0954,3.8769,13.44,0.8293,1.13,17.49
5,Nivel 2 (Feature Eng.),2,Estándar,LightGBM_HistGBM,34,3.1706,0.0193,3.9821,0.8199,3.1066,3.9077,13.57,0.8266,1.13,5.13
6,Nivel 2 (Feature Eng.),2,Estándar,XGBoost_GBDT,34,3.2632,0.0235,4.0972,0.8094,3.2195,4.0474,14.15,0.8139,1.40,25.32
7,Nivel 2 (Feature Eng.),2,Estándar,Ridge,34,4.8116,0.0365,6.0618,0.5827,4.7869,6.0377,21.23,0.5859,9.37,0.28
8,Nivel 3 (Híbrido A+C),3,Global + Especialista Cluster,Hybrid_Cluster_HistGBM,41,3.1655,0.0200,3.9742,0.8206,3.1002,3.9002,13.55,0.8272,1.13,13.86
9,Nivel 3 (A: FE + Clusters),3,Estándar,RandomForest,41,3.1854,0.0325,4.0094,0.8174,3.1377,3.9393,13.65,0.8237,1.31,35.43


## 4. Optimización de Hiperparámetros y Serialización del Modelo Ganador

Se evalúa  mediante Validación Cruzada 5-Fold en `train` para optimizar `max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`, `learning_rate` y el peso de mezcla `global_weight` de `ClusterSpecializedHybridRegressor`. La configuración ganadora se reentrena sobre `train` completo y se exporta a `models/best_eta_model.joblib`.

In [4]:
best_model, df_tuning, tuning_info = tune_and_fit_best_model(
    df_train,
    df_test,
    cv_splits=5,
    random_state=RANDOM_STATE,
    model_path='../models/best_eta_model.joblib',
    preprocessor_path='../models/preprocessor_reg.joblib',
    verbose=True,
)

df_tuning.to_csv(reports_dir / 'tuning_resultados.csv', index=False)

print('\n--- Ranking de configuraciones en 5-Fold CV (train) ---')
display(df_tuning)

m_l2 = tuning_info['test_metrics_lvl2_tuned']
m_l3 = tuning_info['test_metrics_lvl3_tuned']
print(f"Configuración ganadora (por CV): {tuning_info['best_config_name']}")
print(
    f"Nivel 2 Tuned (sin clusters) -> Test MAE: {m_l2['MAE']:.4f} min | "
    f"RMSE: {m_l2['RMSE']:.4f} | R2: {m_l2['R2']:.4f}"
)
print(
    f"Nivel 3 Híbrido (A + C)      -> Test MAE: {m_l3['MAE']:.4f} min | "
    f"RMSE: {m_l3['RMSE']:.4f} | R2: {m_l3['R2']:.4f}"
)

loaded_model = joblib.load('../models/best_eta_model.joblib')
loaded_prep = joblib.load('../models/preprocessor_reg.joblib')
assert np.allclose(
    loaded_model.predict(df_test[cols_3].iloc[:25]),
    tuning_info['preds_lvl3_tuned'][:25],
)
assert loaded_prep.transform(df_test[cols_3].iloc[:5]).shape == (5, len(cols_3))
print('Verificación de serialización joblib OK: best_eta_model.joblib y preprocessor_reg.joblib listos.')

[Tuning Base_Leaf25_w0.70] CV MAE=3.1655 | CV RMSE=3.9742 | CV R2=0.8206


[Tuning Reg_Leaf25_w0.75] CV MAE=3.1547 | CV RMSE=3.9605 | CV R2=0.8219


[Tuning Tuned_Leaf31_w0.80] CV MAE=3.1469 | CV RMSE=3.9498 | CV R2=0.8228



--- Ranking de configuraciones en 5-Fold CV (train) ---


,Config,global_weight,global_max_leaf_nodes,global_min_samples_leaf,cluster_max_leaf_nodes,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_R2_mean
0,Tuned_Leaf31_w0.80,0.80,31,50,25,3.1469,0.0266,3.9498,0.8228
1,Reg_Leaf25_w0.75,0.75,28,55,22,3.1547,0.0244,3.9605,0.8219
2,Base_Leaf25_w0.70,0.70,25,60,20,3.1655,0.0200,3.9742,0.8206


Configuración ganadora (por CV): Tuned_Leaf31_w0.80
Nivel 2 Tuned (sin clusters) -> Test MAE: 3.0879 min | RMSE: 3.8787 | R2: 0.8291
Nivel 3 Híbrido (A + C)      -> Test MAE: 3.0837 min | RMSE: 3.8728 | R2: 0.8296
Verificación de serialización joblib OK: best_eta_model.joblib y preprocessor_reg.joblib listos.
